# Path Finder - Starts and travel to all defined cities and return

## Problem solving by Uninformed & Informed Search

Things to follow
1.	Use appropriate data structures to represent the graph and the path using python libraries
2.	Provide proper documentation
3.	Find the path and print it

Given below are the prominent cities of Ukraine. Now the country is in a crisis due to war, and as an expert in AI, you have been requested to help supply goods to various places using an intelligent agent. The picture below gives the map of the cities connected by paths that are safe to travel. You have been given the task to create an agent which can carry goods like medicines, food and other basic supplies to the required places and deliver them. The agent has restrictions of battery time and the agent is required to supply to all the places and return back to the starting point. You are required to find the shortest path for your work to go un-interrupted. The program should be able to take-in the start node dynamically from the user at run time. The edge costs depicted below is an approximation towards the transportation cost between any pair of cities in Ukraine. For heuristic design, consider all the possible paths between any arbitrary node n to the goal node (Starting point). The average of the total transmission cost across all these paths is the heuristic value h(n).


### PATH FINDER Using A* Algorithm

#### Initial Setup of the A* Star Algorithm


In [1]:
#Code Block : Set Initial State (Must handle dynamic inputs)

# 1. Each City represented as a separate Tupple value to create a code and a city name so Node mapping initialize looks good without quote values
K=('K','Keiv')
H=('H','Kharkiv')
O=('O','Odessa')
D=('D','Dnipro')
L=('L','Lyiy')
E=('E','Kherson')
M=('M','Mikolaive')
A=('A','Mariupol')

# 2. List of all the cities in defined Map
cities = [K,H,O,D,L,E,M,A]

# 3. List of all the cities in defined Map
Node_Map={city[0]: city for city in cities}


In [2]:
#Code Block : Set the matrix for transition & cost (as relevant for the given problem)

# 4. Graph representation of the Nodes and its relationships
Nodes = {
    K : [(H,10),(O,5),(L,17),(M,15)],
    H : [(K,10),(O,7)],
    O : [(H,7),(K,5),(D,25)],
    D : [(O,25),(L,4),(E,11)],
    L : [(K,17),(D,4),(E,10),(A,2),(M,12)],
    E : [(L,10),(D,11)],
    M : [(K,15),(L,12)],
    A : [(L,2)]
}
print(Nodes)

LEAF_NODES = []


{('K', 'Keiv'): [(('H', 'Kharkiv'), 10), (('O', 'Odessa'), 5), (('L', 'Lyiy'), 17), (('M', 'Mikolaive'), 15)], ('H', 'Kharkiv'): [(('K', 'Keiv'), 10), (('O', 'Odessa'), 7)], ('O', 'Odessa'): [(('H', 'Kharkiv'), 7), (('K', 'Keiv'), 5), (('D', 'Dnipro'), 25)], ('D', 'Dnipro'): [(('O', 'Odessa'), 25), (('L', 'Lyiy'), 4), (('E', 'Kherson'), 11)], ('L', 'Lyiy'): [(('K', 'Keiv'), 17), (('D', 'Dnipro'), 4), (('E', 'Kherson'), 10), (('A', 'Mariupol'), 2), (('M', 'Mikolaive'), 12)], ('E', 'Kherson'): [(('L', 'Lyiy'), 10), (('D', 'Dnipro'), 11)], ('M', 'Mikolaive'): [(('K', 'Keiv'), 15), (('L', 'Lyiy'), 12)], ('A', 'Mariupol'): [(('L', 'Lyiy'), 2)]}


In [3]:
################################################################################
#
#           Self defined Logger Method to control logging as
#          logging was needed at different level during development
#
################################################################################
VERBOSE=0
DEBUG =1
INFO=2
WARNING =3
ERROR =4
NOTIFY = 5 # A Highest level for Must to display message

LOGGER_MAP = {0:'verbose',1:'debug',2:'info',3:'warning',4:'error',5:'notify'}

CURRENT_LOG_LEVEL = INFO

def logger(log,level=0,err=""):
  if CURRENT_LOG_LEVEL <= level:
    print(log)
  if err!="":
    print(err)

def verbose(log):
  logger(log,VERBOSE)
def debug(log):
  logger(log,DEBUG)
def info(log):
  logger(log,INFO)
def warning(log):
  logger(log,WARNING)
def error(log,err=""):
  logger(log,ERROR,err)
def notify(log):
  logger(log,NOTIFY)


In [4]:
#Code Block : Write function to design the Transition Model/Successor function. Ideally this would be called while search algorithms are implemented

################################################################################
#
#     Overall Algorithm used for this A* algorthm Implementation to start and
#     End in the same node by visiting all node.
#
#     Algorithm used
#
#     1. DFS - Used for Calculating the All path and its avg cost and min cost
#     2. A*  - Used for travelling from one node to another Node
#     3. TSP - Used for directing the A* to travel to all and node and return
#
#     Explanation of the Code defined
#
#     1.  Node   - Each city is defined as a node or a tupple of Code -City Name
#                 K=('K','Keiv')
#     2.  cities - It is an array defining the list of cities
#     3.  Node_Map - It is a map of citycode - cityTupple
#     4.  Nodes  - It is the Graph which defines the Node and its edges
#     5.  Heuristic Calculation
#               - As given in the question to find the average cost of all nodes
#                 path algo as below
#     5.1 getAllPathWithCostByDFS()
#               - This method is a DFS implementation to find all the possible
#                 paths between any two nodes within the Graph Node
#     5.2 getSumOfAllPathCostDivByHops()
#               - This method is to calculate the Sum of all the path cost
#                 divided by the number of hops in the path. Since this did not
#                 result in an admissable hurestic value I had put a minimum of
#                 SumofPathCost/NoOfHops and minOfAllAvailablePaths. This min
#                 function helped in getting the heuristic function admissible.
#     5.3 calculateAvg()
#               - This method is used to calculate the average =sum/count
#     5.4 getMinAveragePathCost()
#               - This method is to calculate the average path cost and
#                 minimum path costof all the path. This will define the
#                 heuristic as per the ask in the Assignment question
#     5.5 calculateMinCostHueristic()
#                - Method to create the Huristic  and min cost dictionary
#     6.  getValidatedStartNode()
#                - Method for getting the start Node input and validate
#     7.  aStarAlgo()
#                 - Method for A* algorithum implementation Mostly as defined in
#                    the webinar sessions
#                 - aStarAlgo()
#                 - get_neighbors()
#     8.  get_neighbors()
#                 - Method to get the neighbours of the given Node in Graph
#                   as defined in the webinar sessions part of A* algo impl
#     9. TSP implementation
#                 - TSP implementation with A* algorithm
#     9.1 getLeafNodeList()
#                 - Method to find the leaf Node in the Graph
#                   So we can exclude them during the TSA processing
#                   Will append after the calculation
#     9.2. calculatePathCost()
#                 - Method to calculate the path cost of the given path
#     9.3. insertLeafNodesInPath()
#                 - Method to insert the leaf node into the
#                   given path of the first bridge node
#     9.4. TSPAlgo()
#                 - Method for to implement the Travelling Salesman
#                   Problem to visit all node and return the starting node
#                   A* alogirthm is internally used for finding
#                   the path to next node
#
#             Problem : The main problem here is the leaf node which will get
#                       the program to get stuck in the leaf node
#             Solution : The implementation puposfully ignore the leaf node
#                       during the traversal. Plan to insert the leaf node
#                       later to the only bridge node
#      10.1 preProcess()
#                 - Method part of Orchestration
#                   It is to calculate the huristic of avg cost
#                   for among all other nodes
#      10.2 process()
#                 - Method part of Orchestration
#                   It is do the actual TSP alog using A* implementation
#      10.3 postProcess()
#                 - Method part of Orchestration
#                   It is to display the Algorithm output info
#                   Startnode - Path - Cost
#      11 Heuristic Verification
#                 - Both the properties of Heuristic Consistency and
#                   admissibility property are verified
#      11.1 isAdmissible()
#                 - Method for check the A* Heuristic admissibility
#      11.2 isConsistent()
#                 - Method for check the A* Heuristic consistency
#      11.3 goalTestCheck()
#                 -Method checks for below conditions
#                   1. All the cities defined in the graph
#                   2. The start node is same as the end node (reached back )
#      12 orchestrator()
#                 - Method for Orchestration of entire
#                   A* based TSP implementation
#      12.1 Calling the orchestrator() Method triggering full implementation
#
################################################################################



################################################################################
#
#     # 5.1 Method to get all path and its cost using DFS for
#               Part of Huristic Calculation Steps
#
################################################################################
def getAllPathWithCostByDFS(Nodes, start, end, path=[], cost=0):
    verbose(f"getAllPathWithCostByDFS {start} {end} {cost}")
    path = path + [start]           #Appending the start node to the Path
    if start == end:                #If goal node return the path and cost
        return [(path, cost)]
    if start not in Nodes:          #If dead node return empty
        return []
    paths = []
    for t in Nodes[start]:          #Get Node with cost as a tupple info
        node, edge_cost = t         #Get Node and traversal edge cost
        if node not in path:
            tCost = cost + edge_cost  # add edge cost to find the traversal cost
                                      # Calling DFS for each node in Node list
            new_paths = getAllPathWithCostByDFS(Nodes, node, end, path, tCost)
            verbose(f"New Path {new_paths}")
            paths.extend(new_paths) # Merging all path as single list
    return paths

################################################################################
#
#     # 5.2 Method to loop and calculate sum of path cost from list of paths
#               Part of Huristic Calculation Steps
#
################################################################################

def getMinSumOfAllPathCostDivByHops(paths_with_costs):
  verbose(f"getMinSumOfAllPathCostDivByHops {paths_with_costs}")
  sumOfAllPathCost = 0
  minOfPathsCost = min(item[1] for item in paths_with_costs)
  for path, cost in paths_with_costs:
      cityNames = [city[1] for city in path]
      verbose(f"Path: {'-> '.join(cityNames)},Total Cost:{cost},Total Hop:{len(cityNames)}")

      #Since the average of the Cost is not admissible dividing by #hops

      # The avg of hueristic is occationally more hence add
      # min of path cost and the avg cost
      sumOfAllPathCost += min((cost/len(cityNames)),minOfPathsCost)
  return minOfPathsCost,sumOfAllPathCost


################################################################################
#
#        # 5.3  Method to calculate average sum/count
#               Part of Huristic Calculation Steps
#
################################################################################

def calculateAvg(sumOfValues,count):
  avg = 0
  if sumOfValues > 0:               # Check to Avoid any division by zero
      avg = sumOfValues /(count) # Average Calculation
      verbose(f"\nAverage Cost of Paths: {avg}")
  return avg;

################################################################################
#
#       # 5.4 Method to calculate the average path cost from any two nodes
#               Part of Huristic Calculation Steps
# IMPORTANT:The average (path cost/hop) is not admissible couple of nodes hence
#           Using min(avg(pathcost/hopcount),(minimum of all pathcost)/hopcount)
#           the hueristic admissible
#
################################################################################

def getMinAveragePathCost(Nodes, start, end):
  verbose(f"getAveragePathCost {start} {end} ")
  pathAndCostArr = getAllPathWithCostByDFS(Nodes, start, end)
  pathCount= len(pathAndCostArr)
  minOfPathsCost,allPathTotalCost= getMinSumOfAllPathCostDivByHops(pathAndCostArr)
  avgPathCost = calculateAvg(allPathTotalCost,pathCount)
  debug(f"\n Start Node : {start}, End Node : {end}, Average Cost of Paths: {avgPathCost}")
  return minOfPathsCost,avgPathCost

################################################################################
#
#     # 5.5 Method to create the Huristic dictionary
#           having Node -> Average Cost
#           Part of Huristic Calculation Steps
#
################################################################################

def calculateMinCostHueristic(Nodes,startNode):
  NodeHValue={}
  NodeMinPathCost={}
  for city in cities:
      if startNode !=city:
        minPathCost,avgPathCost=getMinAveragePathCost(Nodes,startNode,city)
        NodeHValue[city]=round(avgPathCost)
        NodeMinPathCost[city]=minPathCost
      else:
        NodeHValue[city]=0
        NodeMinPathCost[city]= float('inf')
  debug(f"NodeHValue {NodeHValue}")
  debug(f"NodeMinPathCost {NodeMinPathCost}")
  return NodeMinPathCost,NodeHValue

################################################################################
#
#     # 5.6 Method to create the Huristic dictionary
#           having Node -> Average Cost
#           Part of Huristic Calculation Steps between two specific nodes
#
################################################################################

def calculateHueristicBetweenTwoNode(Nodes,startNode,endNode):
  NodeHValue={}
  #for city in cities:
  if startNode !=endNode:
    avgPathCost=getAveragePathCost(Nodes,startNode,endNode)
    NodeHValue[endNode]=round(avgPathCost)
  else:
    NodeHValue[endNode]=0
  return NodeHValue

################################################################################
#
#        # 6. Method for getting the start Node input
#             And getting the string as an input Node
#
################################################################################

def getValidatedStartNode():
  notify(f"CITY LIST {cities}")
  startStr =(input("Enter the Start City Code or City Name :")).upper()
  startNode=None
  if startStr in Node_Map:
    startNode=Node_Map[startStr]
    debug(f"startNode {startNode}")
  else:
    error("Given City Code or City Name NOT found in the list.")
  return startNode;


################################################################################
#
#        # 7. aStarAlgo() Method for A* algorithum
#         as given in the webinar sessions with Minor Modification
#
################################################################################

def aStarAlgo(start_node, stop_node,Graph_nodes,heuristicMap):
    debug(f"start_node {start_node} \n stop_node {stop_node}")
    heuristic=heuristicMap[start_node]
    debug(f"Graph_nodes {Graph_nodes} \n heuristic {heuristic}")
    open_set = set({start_node})
    closed_set = set()
    g = {}               #store distance from starting node
    parents = {}         # parents contains an adjacency map of all nodes
    #distance of starting node from itself is zero
    g[start_node] = 0
    #start_node is root node i.e it has no parent nodes
    #so start_node is set to its own parent node
    parents[start_node] = start_node
    while len(open_set) > 0:
        n = None
        #node with lowest f() is found
        for v in open_set:
            if n is None or g[v] + heuristic[v] < g[n] + heuristic[n]:
                n = v
        if n == stop_node or Graph_nodes[n] is None:
            pass
        else:
            for (m, weight) in get_neighbors(n,Graph_nodes):
                #nodes 'm' not in first and last set are added to first
                #n is set its parent
                if m not in open_set and m not in closed_set:
                    open_set.add(m)
                    parents[m] = n
                    g[m] = g[n] + weight
                #for each node m,compare its distance from start i.e g(m) to the
                #from start through n node
                else:
                    if g[m] > g[n] + weight:
                        #update g(m)
                        g[m] = g[n] + weight
                        #change parent of m to n
                        parents[m] = n
                        #if m in closed set,remove and add to open
                        if m in closed_set:
                            closed_set.remove(m)
                            open_set.add(m)
        if n == None:
            error('Path does not exist!')
            return None

        # if the current node is the stop_node
        # then we begin reconstructin the path from it to the start_node
        if n == stop_node:
            path = []
            while parents[n] != n:
                path.append(n)
                n = parents[n]
            path.append(start_node)
            path.reverse()
            debug('Path found: {}'.format(path))
            return path
        # remove n from the open_list, and add it to closed_list
        # because all of his neighbors were inspected
        open_set.remove(n)
        closed_set.add(n)
    error('Path does not exist!')
    return None

################################################################################
#
#        # 8. get_neighbors() To get the neighbours of the given Node in Graph
#             It is part of A* algorithum
#             as given in the webinar sessions with Minor Modification
#
################################################################################

#define fuction to return neighbor and its distance
#from the passed node
def get_neighbors(v,Graph_nodes):
    if v in Graph_nodes:
        return Graph_nodes[v]
    else:
        return None



################################################################################
#
#         # 9.1. getLeafNodeList() Method to find the leaf Node in the Graph
#               So we can exclude them during the TSA processing
#
################################################################################
def getLeafNodeList(Nodes):
  leafNodeList = []
  for node in Nodes:
    if len(Nodes[node])==0:
      leafNodeList.append(node)
  return leafNodeList


################################################################################
#
#         # 9.2. calculatePathCost() Method to calculate the path cost of the
#                given path
#
################################################################################

# Method to calculate the path cost
def calculatePathCost(path):
    cost = 0
    for i in range(len(path) - 1):
        current_node = path[i]
        next_node = path[i + 1]
        for neighbor, weight in Nodes[current_node]:
            if neighbor == next_node:
                cost += weight
    return cost

################################################################################
#
#         # 9.3. insertLeafNodesInPath() Method to insert the leaf node into the
#                given path of the first bridge node
#
################################################################################

def insertLeafNodesInPath(path, Nodes):
    new_path = []
    for i in range(len(path) - 1):
        currentNode = path[i]
        nextNode = path[i + 1]

        # Check if current or next node is a leaf node
        if currentNode in LEAF_NODES:
            leafNode = currentNode
            # Insert the leaf node between the only neighbor
            bridgeNode = Nodes[leafNode][0][0]
            # Ensure that the leaf node is inserted only in the correct bridge
            if nextNode == bridgeNode:
                new_path.append(currentNode)
                new_path.append(leafNode)  # Inserting leaf node into path
            else:
                new_path.append(leafNode)
                new_path.append(currentNode)
        else:
            new_path.append(currentNode)

    new_path.append(path[-1])  # Add the last node in the path
    return new_path

################################################################################
#
#         # 9.4. TSPAlgo() Method for to implement the Travelling Salesman
#              Problem to visit all node and return the starting node
#              A* alogirthm is internally used for finding the path to next node
#
#             Problem : The main problem here is the leaf node which will get
#                       the program to get stuck in the leaf node
#             Solution : The implementation puposfully ignore the leaf node
#                       during the traversal. Plan to insert the leaf node
#                       later to the only bridge node
#
################################################################################
import sys
def tspAlgo(Nodes,NodeHValueMap,startNode):
  # Initialize visited set and path list
  visited = set()
  visited.add(startNode)
  path = [startNode]
  total_cost = 0

  current_node = startNode
  remaining_nodes = set(Nodes.keys()) - visited

  #Removeing leaf nodes from remaining nodes to avoid them in the initial calculation
  remaining_nodes -= set(LEAF_NODES)

  while remaining_nodes:
      next_node = None
      min_cost = sys.maxsize
      best_path_segment = None

      # For each unvisited non-leaf node, run A* to find the shortest path
      for target_node in remaining_nodes:
          heuristic_map = NodeHValueMap[current_node]
          path_segment = aStarAlgo(current_node, target_node, Nodes, NodeHValueMap)

          if path_segment:
              segmentCost = calculatePathCost(path_segment)
              if segmentCost < min_cost:
                  min_cost = segmentCost
                  next_node = target_node
                  best_path_segment = path_segment

      if next_node is None:
          break

      visited.add(next_node)
      debug("=================================================================")
      debug(f"current_node: {current_node}")
      debug(f"next_node: {next_node}")
      debug(f" best_path_segment: {best_path_segment}")
      path.extend(best_path_segment[1:])
      debug(f" path: {path}")
      total_cost += min_cost
      debug(f" remaining_nodes: {remaining_nodes}")
      remaining_nodes.remove(next_node)
      debug(f" remaining_nodes: {remaining_nodes}")
      current_node = next_node
  debug(f" TSA Algo Path: {path}")
  # Now, add leaf nodes back to the path in the correct positions
  path_with_leaf_nodes = insertLeafNodesInPath(path, Nodes)
  #path_with_leaf_nodes =path
  # Return to the starting node to complete the tour
  return_to_start_path = aStarAlgo(current_node, startNode, Nodes, NodeHValueMap)
  if return_to_start_path:
      total_cost += calculatePathCost(return_to_start_path)
      path_with_leaf_nodes.extend(return_to_start_path[1:])

  return path_with_leaf_nodes, total_cost


################################################################################
#
#         # 10.1 displayMapAsmatrix() Method part of display stratergy to
#                 to display map as beautiful table
#
################################################################################

import pandas as pd  # For creating tables from Map using Pandas
def displayMapAsmatrix(matrix, title):
    df = pd.DataFrame(matrix)
    notify(title)
    notify(df.to_string())  # Use to_string() for formatted output

################################################################################
#
#         # 10.2 preProcess() Method part of Orchestration
#                It is to calculate the huristic of avg cost
#                for among all other nodes
#
################################################################################

def preProcess():
  NodeMinCostMap = {}
  NodeHValueMap = {}
  for node in cities:
    NodeMinCost,NodeHValue = calculateMinCostHueristic(Nodes,node)
    NodeHValueMap[node] = NodeHValue
    NodeMinCostMap[node]= NodeMinCost
  displayMapAsmatrix(NodeHValueMap,"Minimum Cost Matrix:")
  displayMapAsmatrix(NodeMinCostMap,"Heuristic Value Matrix:")
  return NodeMinCostMap,NodeHValueMap

################################################################################
#
#         # 10.3 process() Method part of Orchestration
#                It is do the actual TSP alog using A* implementation
#
################################################################################

def process(NodeHValueMap,startNode):
  full_path, min_cost = tspAlgo(Nodes,NodeHValueMap,startNode)
  return full_path,min_cost

################################################################################
#
#         # 10.4 postProcess() Method part of Orchestration
#                It is to display the Algorithm output info
#                Startnode - Path - Cost
#
################################################################################

def postProcess(Nodes,NodeHValueMap,startNode,full_path,min_cost):
  notify("-----------------------------------------------------------------")
  notify(f"Start Node : {startNode}")
  if full_path:
      city_names = [city[1] for city in full_path]
      pathOutput = " Path:", " -> ".join(city_names)

      notify("-----------------------------------------------------------------")
      notify("Optimal Path:")
      notify("-----------------------------------------------------------------")
      notify(" -> ".join([city[1] for city in full_path])) # Print city names
      notify(f"\nTotal Cost:{min_cost}")
  else:
      notify("No TSP path found.")
  notify("-----------------------------------------------------------------")



################################################################################
#
#         # 11.1 isAdmissible() Method for check the A* Heuristic admissibility
#
################################################################################

def isAdmissible(NodeMinCostMap, NodeHValueMap):
  for startNode in cities:
    for endNode in cities:
      shortestCost = (NodeMinCostMap[startNode])[endNode]
      if (NodeHValueMap[startNode]).get(endNode) > shortestCost:
          error(f"Heuristic is inadmissible from startNode {startNode} endNode {endNode}: Estimated cost ({(NodeHValueMap[startNode]).get(endNode)}) > True cost ({shortestCost})")
          return False
  return True

################################################################################
#
#         # 11.2 isConsistent() Method for check the A* Heuristic consistency
#
################################################################################

def isConsistent(NodeHValueMap):
  for node in cities:
      nodeHeuristic = NodeHValueMap[node]
      for neighbor, weight in Nodes[node]:
          neighborHeuristic = NodeHValueMap[node]
          if node in neighborHeuristic and neighbor in nodeHeuristic:
              if nodeHeuristic[neighbor] > weight + neighborHeuristic[node]:
                  error(f"Heuristic is inconsistent: h({node}) ({nodeHeuristic[neighbor]}) > c({node},{neighbor}) + h({neighbor}) ({weight} + {neighborHeuristic[node]})")
                  return False
  return True


In [5]:
#Code block : Write fucntion to handle goal test (Must handle dynamic inputs). Ideally this would be called while search algorithms are implemented


################################################################################
#
#         # 11.3 goalTestCheck() Method checks for below conditions
#                 1. All the cities defined in the graph
#                 2. The start node is same as the end node (reached back )
#
################################################################################

def goalTestCheck(path):
  # Checking all the cities are visited atleast once
  for node in cities:
    if node not in path:
      error(f"Error > City {node} not visited in the {path}. Hence Goal Test Failed")
      return False
  # Checking if the start node and final node are same
  if(path[0] != path[-1]):
    error(f"Error > Path  Start node {path[0]} not same as the end node  {path[-1]}. Hence Goal Test Failed")
    return False
  return True


In [6]:
################################################################################
#
#         # 12 orchestrator() Method for Orchestration of entire
#               A* based TSP implementation
#
################################################################################

def orchestrator(startNode):
  notify("####################################################################")
  notify("#                                                                  #")
  notify("#      Implementation of the A* Algoirthm using TSP solution       #")
  notify("#                                                                  #")
  notify(f"#                   Logger Level - {LOGGER_MAP[CURRENT_LOG_LEVEL]}                            #")
  notify("####################################################################")


  LEAF_NODES = getLeafNodeList(Nodes)


  NodeMinCostMap,NodeHValueMap = preProcess()

  admissible = isAdmissible(NodeMinCostMap, NodeHValueMap)
  notify(f"Is Admissible: {admissible}")
  consistent = isConsistent(NodeHValueMap)
  notify(f"Is Consistent: {consistent}")
  if admissible :
    if consistent :
      #Starting Time and space Complexity
      import time
      import tracemalloc
      tracemalloc.start()  # Start memory tracing
      startTime = time.perf_counter()

      #Actual Algorithm of TSP using A* algorithm
      full_path, min_cost = process(NodeHValueMap,startNode)

      endTime = time.perf_counter()
      timeTaken = endTime - startTime
      current, peak = tracemalloc.get_traced_memory()
      tracemalloc.stop()
      #Ending Time and space Complexity
      goalCheck = goalTestCheck(full_path)
      notify(f"Is Goal Test Check :{goalCheck} ie Visited all nodes and returned to start node {startNode}")
      postProcess(Nodes,NodeHValueMap,startNode,full_path,min_cost)

      notify(f"Time taken: {timeTaken:.6f} seconds")
      notify(f"Memory used: {current / 10**6:.5f} MB (current), {peak / 10**6:.5f} MB (peak)")  # In MB

#### DYNAMIC INPUT

IMPORTANT : Dynamic Input must be got in this section. Display the possible states to choose from:
This is applicable for all the relevent problems as mentioned in the question.

#### Calling the search algorithms
(For bidirectional search in below sections first part can be used as per Hint provided. Under second section other combinations as per Hint or your choice of 2 algorithms can be called .As an analyst suggest suitable approximation in the comparitive analysis section)

In [7]:
#Invoke algorithm 2 (Should Print the solution, path, cost etc., (As mentioned in the problem))
startNode = getValidatedStartNode()
CURRENT_LOG_LEVEL = INFO
orchestrator(startNode)


CITY LIST [('K', 'Keiv'), ('H', 'Kharkiv'), ('O', 'Odessa'), ('D', 'Dnipro'), ('L', 'Lyiy'), ('E', 'Kherson'), ('M', 'Mikolaive'), ('A', 'Mariupol')]
Enter the Start City Code or City Name :M
####################################################################
#                                                                  #
#      Implementation of the A* Algoirthm using TSP solution       #
#                                                                  #
#                   Logger Level - info                            #
####################################################################
Minimum Cost Matrix:
               K       H      O      D    L       E         M        A
            Keiv Kharkiv Odessa Dnipro Lyiy Kherson Mikolaive Mariupol
K Keiv         0       8      5      9    9       9        10        8
H Kharkiv      8       0      6      9    9       9        10        8
O Odessa       5       6      0      9    9       9        10        8
D Dnipro       9  

#### Calling the search algorithms with Detailed Logs

In [8]:
# Can run the code with DEBUG log for detail loggin
CURRENT_LOG_LEVEL = DEBUG # For More detailed logs can add VERBOSE
notify("\n\n\n\n\n Calling the Same A* Algo with DEBUG Log for more details ")
orchestrator(startNode)






 Calling the Same A* Algo with DEBUG Log for more details 
####################################################################
#                                                                  #
#      Implementation of the A* Algoirthm using TSP solution       #
#                                                                  #
#                   Logger Level - debug                            #
####################################################################

 Start Node : ('K', 'Keiv'), End Node : ('H', 'Kharkiv'), Average Cost of Paths: 8.166666666666666

 Start Node : ('K', 'Keiv'), End Node : ('O', 'Odessa'), Average Cost of Paths: 4.583333333333333

 Start Node : ('K', 'Keiv'), End Node : ('D', 'Dnipro'), Average Cost of Paths: 9.058333333333334

 Start Node : ('K', 'Keiv'), End Node : ('L', 'Lyiy'), Average Cost of Paths: 9.316666666666666

 Start Node : ('K', 'Keiv'), End Node : ('E', 'Kherson'), Average Cost of Paths: 9.204166666666667

 Start Node : ('K', 'Keiv

#### **Summary**

The search algorithm - **A* algorithm** was tested all valid routes (starting and ending in the same city) and was able to find the best path based on the heuristic function.